# Neo4j Simple Connection Test

Quick validation of Neo4j connectivity: network (TCP) and Python driver.

---

## Configuration

Set your Neo4j Aura credentials below.

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these two sections only
# =============================================================================

# --- Neo4j Aura ---
NEO4J_URI = "neo4j+s://123456.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

# --- Databricks Unity Catalog ---
UC_CATALOG = "<catalog>"
UC_SCHEMA = "neo4j_getting_started"
UC_VOLUME = "data"
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
# If your catalog name contains hyphens (e.g., "uc-w-neo4j"), Spark SQL requires
# backtick quoting. Without it you'll get:
#   [INVALID_IDENTIFIER] The unquoted identifier uc-w-neo4j is invalid
#   and must be back quoted as: `uc-w-neo4j`.
# FQN handles this automatically.
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/neo4j"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Section 1: Network Connectivity Test (TCP Layer)

**Expected Result**: PASS - Proves network path is open.

In [ ]:
# TCP connectivity test using netcat
import subprocess
import time

print("=" * 60)
print("TEST: Network Connectivity (TCP)")
print("=" * 60)
print(f"\nTarget: {NEO4J_HOST}:7687 (Bolt protocol port)")
print("Testing: Can we reach Neo4j at the network level?")

try:
    start_time = time.time()
    result = subprocess.run(
        ['nc', '-zv', NEO4J_HOST, '7687'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10
    )
    elapsed = (time.time() - start_time) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print("\n" + "=" * 60)
        print(">>> CONNECTIVITY VERIFIED <<<")
        print("=" * 60)
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        print(f"\nConnection Details:")
        print(f"  - Host: {NEO4J_HOST}")
        print(f"  - Port: 7687 (Bolt)")
        print(f"  - TCP Latency: {elapsed:.1f}ms")
        if output:
            print(f"  - Raw Output: {output}")
        print("\n" + "-" * 60)
        print("RESULT: Network path to Neo4j is OPEN")
        print("        Firewall rules allow Bolt protocol traffic")
        print("-" * 60)
        print("\nStatus: PASS")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_HOST}:7687 - check firewall rules")
        print(f"Details: {output}")
        print("\nStatus: FAIL")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")
    print("\nStatus: FAIL")

---

## Section 2: Neo4j Python Driver Test

**Expected Result**: PASS - Proves credentials work and Neo4j is accessible.

In [ ]:
# Test Neo4j Python driver connectivity
import time

print("=" * 60)
print("TEST: Neo4j Python Driver")
print("=" * 60)
print(f"\nTarget: {NEO4J_URI}")
print("Testing: Can we authenticate and execute queries via Bolt protocol?")

from neo4j import GraphDatabase

try:
    start_time = time.time()
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    
    # Verify connectivity
    driver.verify_connectivity()
    connect_time = (time.time() - start_time) * 1000
    
    print("\n" + "=" * 60)
    print(">>> AUTHENTICATION SUCCESSFUL <<<")
    print("=" * 60)
    print(f"\n[PASS] Driver connected and authenticated in {connect_time:.1f}ms")
    
    # Test simple query
    with driver.session() as session:
        query_start = time.time()
        result = session.run("RETURN 1 AS test")
        record = result.single()
        query_time = (time.time() - query_start) * 1000
        print(f"[PASS] Query executed: RETURN 1 = {record['test']} ({query_time:.1f}ms)")
        
        # Get Neo4j version
        result = session.run("CALL dbms.components() YIELD name, versions RETURN name, versions")
        neo4j_info = []
        for record in result:
            neo4j_info.append(f"{record['name']} {record['versions']}")
    
    total_time = (time.time() - start_time) * 1000
    driver.close()
    
    print(f"\nConnection Details:")
    print(f"  - URI: {NEO4J_URI}")
    print(f"  - User: {NEO4J_USERNAME}")
    print(f"  - Neo4j Server: {', '.join(neo4j_info)}")
    print(f"  - Connection Time: {connect_time:.1f}ms")
    print(f"  - Total Test Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: Neo4j Python Driver connection WORKING")
    print("        Credentials valid, Bolt protocol functional")
    print("-" * 60)
    print("\nStatus: PASS")
    
except Exception as e:
    print(f"\n[FAIL] Connection failed: {e}")
    print("\nStatus: FAIL")

---

## Section 3: Create Sample Test Data

Creates a small knowledge graph of companies, products, and competitive relationships to validate write operations and provide data for query testing.

In [ ]:
# Create sample test data
from neo4j import GraphDatabase

print("=" * 60)
print("SETUP: Creating Sample Test Data")
print("=" * 60)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    with driver.session() as session:
        # Clear any existing test data
        session.run("MATCH (n) WHERE n:Company OR n:Product DETACH DELETE n")
        print("\n[OK] Cleared existing Company/Product nodes")

        # Create Company nodes
        session.run("""
            CREATE (:Company {companyId: 'C001', name: 'Amazon.com, Inc.', ticker: 'AMZN'})
            CREATE (:Company {companyId: 'C002', name: 'Apple Inc.', ticker: 'AAPL'})
            CREATE (:Company {companyId: 'C003', name: 'Microsoft Corporation', ticker: 'MSFT'})
            CREATE (:Company {companyId: 'C004', name: 'NVIDIA Corporation', ticker: 'NVDA'})
        """)
        print("[OK] Created 4 Company nodes")

        # Create Product nodes
        session.run("""
            CREATE (:Product {productId: 'P011', name: 'AWS'})
            CREATE (:Product {productId: 'P018', name: 'Amazon Prime'})
            CREATE (:Product {productId: 'P020', name: 'App Store'})
            CREATE (:Product {productId: 'P030', name: 'Apple Vision Pro'})
            CREATE (:Product {productId: 'P036', name: 'Azure'})
            CREATE (:Product {productId: 'P037', name: 'Azure AI'})
            CREATE (:Product {productId: 'P002', name: 'A100 Integrated Circuit'})
            CREATE (:Product {productId: 'P006', name: 'AI Cloud Services'})
        """)
        print("[OK] Created 8 Product nodes")

        # Create OFFERS relationships (companies to products)
        session.run("""
            MATCH (c:Company {companyId: 'C001'}), (p:Product {productId: 'P011'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C001'}), (p:Product {productId: 'P018'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C002'}), (p:Product {productId: 'P020'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C002'}), (p:Product {productId: 'P030'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C003'}), (p:Product {productId: 'P036'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C003'}), (p:Product {productId: 'P037'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C004'}), (p:Product {productId: 'P002'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        session.run("""
            MATCH (c:Company {companyId: 'C004'}), (p:Product {productId: 'P006'})
            CREATE (c)-[:OFFERS]->(p)
        """)
        print("[OK] Created 8 OFFERS relationships")

        # Create COMPETES_WITH relationships
        session.run("""
            MATCH (a:Company {companyId: 'C001'}), (b:Company {companyId: 'C003'})
            CREATE (a)-[:COMPETES_WITH]->(b)
        """)
        session.run("""
            MATCH (a:Company {companyId: 'C002'}), (b:Company {companyId: 'C003'})
            CREATE (a)-[:COMPETES_WITH]->(b)
        """)
        session.run("""
            MATCH (a:Company {companyId: 'C003'}), (b:Company {companyId: 'C001'})
            CREATE (a)-[:COMPETES_WITH]->(b)
        """)
        session.run("""
            MATCH (a:Company {companyId: 'C003'}), (b:Company {companyId: 'C002'})
            CREATE (a)-[:COMPETES_WITH]->(b)
        """)
        session.run("""
            MATCH (a:Company {companyId: 'C004'}), (b:Company {companyId: 'C001'})
            CREATE (a)-[:COMPETES_WITH]->(b)
        """)
        session.run("""
            MATCH (a:Company {companyId: 'C004'}), (b:Company {companyId: 'C003'})
            CREATE (a)-[:COMPETES_WITH]->(b)
        """)
        print("[OK] Created 6 COMPETES_WITH relationships")

        # Verify counts
        result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY label")
        print(f"\nNode counts:")
        for record in result:
            print(f"  - {record['label']}: {record['count']}")

        result = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY type")
        print(f"\nRelationship counts:")
        for record in result:
            print(f"  - {record['type']}: {record['count']}")

    print("\n" + "-" * 60)
    print("RESULT: Sample test data created successfully")
    print("        4 Company nodes, 8 Product nodes")
    print("        8 OFFERS relationships, 6 COMPETES_WITH relationships")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Error creating test data: {e}")
    print("\nStatus: FAIL")
finally:
    driver.close()

---

## Section 4: Unity Catalog JDBC Connection

Create and test the UC JDBC connection. The SafeSpark sandbox wraps the JDBC driver in an isolated JVM.

**Note:** `java_dependencies` only accepts UC Volume paths.

In [ ]:
# Create Unity Catalog JDBC Connection
import time

print("=" * 60)
print("SETUP: Creating UC JDBC Connection")
print("=" * 60)
print(f"\nConnection Name: {UC_CONNECTION_NAME}")
print(f"JDBC URL: {NEO4J_JDBC_URL_SQL}")

spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")

create_sql = f"""
CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{NEO4J_JDBC_URL_SQL}',
  user '{NEO4J_USERNAME}',
  password '{NEO4J_PASSWORD}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

try:
    start_time = time.time()
    spark.sql(create_sql)
    elapsed = (time.time() - start_time) * 1000

    print(f"\n[PASS] Connection created in {elapsed:.1f}ms")

    # Verify connection
    spark.sql(f"DESCRIBE CONNECTION {UC_CONNECTION_NAME}").show(truncate=False)

    # Test with SELECT 1
    print("Testing UC JDBC with SELECT 1...")
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", "SELECT 1 AS test") \
        .option("customSchema", "test INT") \
        .load()

    results = df.collect()
    elapsed = (time.time() - start_time) * 1000

    print(f"[PASS] UC JDBC query returned: {results[0]['test']} ({elapsed:.1f}ms)")

    print("\n" + "-" * 60)
    print("RESULT: UC JDBC connection created and verified")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("\nStatus: FAIL")

---

## Section 5: Query Sample Data via UC JDBC

Run SQL queries against the Neo4j test data through the Unity Catalog JDBC connection. SQL is automatically translated to Cypher by the connector.

**Note**: `customSchema` is required because Neo4j's JDBC driver returns `NullType` during Spark's schema inference.

In [ ]:
# Query sample data via UC JDBC
import time

print("=" * 60)
print("TEST: Query Sample Data via UC JDBC")
print("=" * 60)

def run_jdbc_query(description, custom_schema, query=None, dbtable=None):
    """Run a SQL query or dbtable read through the UC JDBC connection.

    Use `query` for aggregates (COUNT, GROUP BY). Use `dbtable` for
    non-aggregate reads — it avoids Spark's subquery wrapping which the
    Neo4j SQL-to-Cypher translator cannot handle for non-aggregate SELECT.
    """
    print(f"\n--- {description} ---")
    if query:
        print(f"SQL (query): {query}")
    else:
        print(f"Label (dbtable): {dbtable}")
    try:
        start_time = time.time()
        reader = spark.read.format("jdbc") \
            .option("databricks.connection", UC_CONNECTION_NAME) \
            .option("customSchema", custom_schema)
        if query:
            reader = reader.option("query", query)
        else:
            reader = reader.option("dbtable", dbtable)
        df = reader.load()
        elapsed = (time.time() - start_time) * 1000
        print(f"[PASS] Query returned in {elapsed:.1f}ms")
        df.show(truncate=False)
        return df
    except Exception as e:
        print(f"[FAIL] {e}")
        return None

# Query 1: Count all Company nodes (aggregate — uses query option)
run_jdbc_query(
    "Count Companies",
    custom_schema="company_count LONG",
    query="SELECT COUNT(*) AS company_count FROM Company",
)

# Query 2: List all companies (non-aggregate — uses dbtable option)
run_jdbc_query(
    "All Companies",
    custom_schema="companyId STRING, name STRING, ticker STRING",
    dbtable="Company",
)

# Query 3: Count all Product nodes (aggregate — uses query option)
run_jdbc_query(
    "Count Products",
    custom_schema="product_count LONG",
    query="SELECT COUNT(*) AS product_count FROM Product",
)

# Query 4: All products (non-aggregate — uses dbtable option)
run_jdbc_query(
    "All Products",
    custom_schema="productId STRING, name STRING",
    dbtable="Product",
)

print("\n" + "-" * 60)
print("RESULT: UC JDBC queries executed successfully")
print("        SQL translated to Cypher by the connector")
print("-" * 60)
print("\nStatus: PASS")